In [0]:
%run /Workspace/Users/nk1956663@gmail.com/Functions

In [0]:
connect()

In [0]:
# Reading Products file 

bronze_path_products = "/Volumes/proj_databricks/retail/bronze/Products"


products_delta = spark.read \
    .format("delta") \
    .load(bronze_path_products)

display(products_delta)



In [0]:
# Before Transfomation Count of products table 


df2 = products_delta.count()
display(df2)

In [0]:
# Transformation on Products file


from pyspark.sql import SparkSession
from pyspark.sql.functions import col, coalesce, lit

# 1. Read Bronze Data
df_products_bronze = spark.read.format("delta").load("/Volumes/proj_databricks/retail/bronze/Products")

# 2. Transformations for Silver Layer
df_products_silver = (
    df_products_bronze
    # A. Deduplication
    .dropDuplicates(["product_id"])
    
    # B. Filter Invalid/Negative Prices
    .filter((col("selling_price") > 0) & (col("selling_price") >= col("cost_price")))
    
    # C. Handle Missing / Null Values
    .withColumn("category", coalesce(col("category"), lit("Unknown")))
    .withColumn("brand", coalesce(col("brand"), lit("Unknown")))
    .withColumn("stock_quantity", coalesce(col("stock_quantity"), lit(0)))
)








In [0]:
# After Transformation Product file Rows count -

df4 = df_products_silver.count()
display(df4)

In [0]:
# Saving Products File in Silver Layer


Silver_path_products = "/Volumes/proj_databricks/retail/silver/Products_silver"


df_products_silver.write \
.format("delta") \
.mode("append") \
.save(Silver_path_products)

In [0]:
# 

In [0]:
# Reading Customer file 

bronze_path_customer = "/Volumes/proj_databricks/retail/bronze/Customers"


customer_delta = spark.read \
    .format("delta") \
    .load(bronze_path_customer)

display(customer_delta)


In [0]:
# Before Transformation Rows count - 

dff = customer_delta.count()
display(dff)

In [0]:
# Transformation on customer file


from pyspark.sql.functions import col, trim, coalesce, lit, when

# 1. Read Bronze JSON Data
df_customers_bronze = spark.read.format("delta").load("/Volumes/proj_databricks/retail/bronze/Customers")

# 2. Transformations for Silver Layer
email_regex = "^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}$"

df_customers_silver = (
    df_customers_bronze
    # A. Flattening Nested JSON & Trimming Spaces
    .select(
        col("customer_id"),
        trim(col("name.first")).alias("first_name"),
        trim(col("name.last")).alias("last_name"),
        col("contact.email").alias("email"),
        coalesce(col("contact.phone"), lit("N/A")).alias("phone"),
        coalesce(col("address.city"), lit("Unknown")).alias("city"),
        col("address.state").alias("state"),
        col("address.country").alias("country"),
        col("signup_date"),
        col("customer_type"),
        # D. Invalid Age Handling (-5 to NULL)
        when(col("age") > 0, col("age")).otherwise(None).alias("age")
    )
    # C. Invalid Email Filter
    .filter(col("email").rlike(email_regex))
    # Deduplication
    .dropDuplicates(["customer_id"])
)



In [0]:
# After Transformation Customer file Rows count -

df6 = df_customers_silver.count()
display(df6)

In [0]:
# Saving Customer File in Silver Layer


Silver_path_customer = "/Volumes/proj_databricks/retail/silver/Customer_silver"


df_customers_silver.write \
.format("delta") \
.mode("append") \
.save(Silver_path_customer)

In [0]:
# Path for Orders File

bronze_path = "/Volumes/proj_databricks/retail/bronze/Orders"

In [0]:
# Reading Orders File


Orders_delta = spark.read \
    .format("delta") \
    .load(bronze_path)

display(Orders_delta)

In [0]:
# Before Transformation Rows count - 5.5 lakh +

df = Orders_delta.count()
display(df)

In [0]:
# Transformation on Orders File

from pyspark.sql.functions import col, coalesce, lit, try_to_date, trim, initcap, regexp_replace

# 1. Read Bronze Delta Data
df_orders_bronze = spark.read.format("delta").load("/Volumes/proj_databricks/retail/bronze/Orders")

# 2. Initial Transformations & Data Hygiene
df_orders_temp = (
    df_orders_bronze
    # A. Deduplication
    .dropDuplicates(["order_id"])
    
    # B. Inconsistent Date Format Fix (Safe parsing using try_to_date + trim)
    .withColumn(
        "order_date", 
        coalesce(
            try_to_date(trim(col("order_date")), "dd/MM/yyyy"),
            try_to_date(trim(col("order_date")), "yyyy-MM-dd")
        )
    )
    .withColumn(
        "delivery_date", 
        coalesce(
            try_to_date(trim(col("delivery_date")), "yyyy-MM-dd"),
            try_to_date(trim(col("delivery_date")), "dd/MM/yyyy")
        )
    )
    
    # C. Filter Negative Quantities
    .filter(col("quantity").cast("int") > 0)
    
    # D. Null Shipping Cost Handling
    .withColumn("shipping_cost", coalesce(col("shipping_cost").cast("double"), lit(0.0)))
    
    # E. Standardize Payment Method Case (e.g., 'cash_on_delivery' -> 'Cash On Delivery')
    .withColumn(
        "payment_method", 
        initcap(regexp_replace(col("payment_method"), "_", " "))
    )
    
    # Casting correct data types
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("unit_price", col("unit_price").cast("double"))
    .withColumn("discount_percent", col("discount_percent").cast("double"))
    .withColumn("discount_amount", col("discount_amount").cast("double"))
    .withColumn("tax_amount", col("tax_amount").cast("double"))
)

# 3. Referential Integrity Check (Remove Orphan Keys using Left Semi Join)
df_orders_silver = (
    df_orders_temp
    .join(df_customers_silver, "customer_id", "left_semi")
    .join(df_products_silver, "product_id", "left_semi")
)

In [0]:
# After Transformation Order file Rows Count


df7 = df_orders_silver.count()
display(df7)

In [0]:
# Saving Customer File in Silver Layer


Silver_path_orders = "/Volumes/proj_databricks/retail/silver/Orders_silver"


df_orders_silver.write \
.format("delta") \
.mode("append") \
.save(Silver_path_orders)